In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import pandas as pd

# Autenticação e conexão ao Workspace
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential)

# Carregando o Data Asset (ajuste o nome se mudou na criação)
data_asset = ml_client.data.get("breast_cancer", version="1")
df = pd.read_csv(data_asset.path)

# Limpeza inicial específica
df = df.drop(columns=['Unnamed: 32', 'id'], errors='ignore')
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

print(f"Dados carregados! Formato: {df.shape}")
df.head()

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
Overriding of current TracerProvider is not allowed
Overriding

Dados carregados! Formato: (569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

# IMPORTANDO do seu arquivo auxiliar!
from ag_functions import create_individual, crossover, mutate

# Preparando os dados
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Função Fitness (Aptidão) - Baseada em F1-Score
def fitness(individual):
    model = RandomForestClassifier(
        n_estimators=int(individual[0]),
        max_depth=int(individual[1]),
        min_samples_split=individual[2],
        random_state=42
    )
    # Retorna a média de 3-fold cross validation
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='f1')
    return scores.mean()

print("Dados preparados e Função Fitness carregada!")

Dados preparados e Função Fitness carregada!


In [3]:
import time

def run_genetic_experiment(pop_size, generations, mutation_rate, exp_name):
    print(f"\n--- Iniciando {exp_name} ---")
    start_time = time.time()
    
    # Inicialização da população
    population = [create_individual() for _ in range(pop_size)]
    history = []

    for gen in range(generations):
        # Avaliação de Fitness
        pop_fitness = [(ind, fitness(ind)) for ind in population]
        pop_fitness.sort(key=lambda x: x[1], reverse=True)
        
        best_current_score = pop_fitness[0][1]
        history.append(best_current_score)
        
        # Seleção
        best_individuals = [x[0] for x in pop_fitness[:pop_size//2]]
        
        # Cruzamento e Mutação
        next_gen = best_individuals.copy()
        while len(next_gen) < pop_size:
            p1, p2 = np.random.choice(len(best_individuals), 2)
            child = crossover(best_individuals[p1], best_individuals[p2])
            # Aplicando a taxa de mutação variável do experimento
            next_gen.append(mutate(child, prob=mutation_rate))
            
        population = next_gen
        if gen % 2 == 0:
            print(f"Geração {gen}: Melhor F1 = {best_current_score:.4f}")

    end_time = time.time()
    duration = end_time - start_time
    
    return {
        "nome": exp_name,
        "pop_size": pop_size,
        "mut_rate": mutation_rate,
        "best_score": pop_fitness[0][1],
        "best_params": pop_fitness[0][0],
        "tempo": duration
    }

In [4]:
import mlflow
import numpy as np

# 1. Definir o nome do experimento na Azure
mlflow.set_experiment("Otimizacao_Diagnostico_AG")

def executar_experimento_com_log(pop_size, generations, mutation_rate, exp_name):
    # Inicia o rastreamento na Azure
    with mlflow.start_run(run_name=exp_name):
        # Logar os Hiperparâmetros do AG (Entradas)
        mlflow.log_param("tamanho_populacao", pop_size)
        mlflow.log_param("geracoes", generations)
        mlflow.log_param("taxa_mutacao", mutation_rate)
        
        # Executa a lógica que já criamos
        resultado = run_genetic_experiment(pop_size, generations, mutation_rate, exp_name)
        
        # Logar as Métricas de Sucesso (Saídas)
        mlflow.log_metric("melhor_f1_score", resultado["best_score"])
        mlflow.log_metric("tempo_execucao_seg", resultado["tempo"])
        
        # Logar os melhores parâmetros encontrados como artefato ou tag
        mlflow.set_tag("melhores_params", str(resultado["best_params"]))
        
        print(f"✅ {exp_name} finalizado e logado na Azure!")
        return resultado

# Executando os 3 experimentos obrigatórios com rastreamento
exp1 = executar_experimento_com_log(10, 10, 0.05, "AG_Conservador")
exp2 = executar_experimento_com_log(15, 10, 0.4, "AG_Exploratorio")
exp3 = executar_experimento_com_log(40, 10, 0.1, "AG_Robusto")


--- Iniciando AG_Conservador ---
Geração 0: Melhor F1 = 0.9299
Geração 2: Melhor F1 = 0.9299
Geração 4: Melhor F1 = 0.9332
Geração 6: Melhor F1 = 0.9364
Geração 8: Melhor F1 = 0.9364
✅ AG_Conservador finalizado e logado na Azure!
🏃 View run AG_Conservador at: https://brazilsouth.api.azureml.ms/mlflow/v2.0/subscriptions/d91397ec-f486-481e-a7ba-4e64f05b7962/resourceGroups/meugrupo/providers/Microsoft.MachineLearningServices/workspaces/tech-challenge-2/#/experiments/d6fab33f-79b9-4c9b-bcf1-d5658af23438/runs/e9db13c1-c739-420c-b725-db15c85b1e2b
🧪 View experiment at: https://brazilsouth.api.azureml.ms/mlflow/v2.0/subscriptions/d91397ec-f486-481e-a7ba-4e64f05b7962/resourceGroups/meugrupo/providers/Microsoft.MachineLearningServices/workspaces/tech-challenge-2/#/experiments/d6fab33f-79b9-4c9b-bcf1-d5658af23438

--- Iniciando AG_Exploratorio ---
Geração 0: Melhor F1 = 0.9244
Geração 2: Melhor F1 = 0.9332
Geração 4: Melhor F1 = 0.9332
Geração 6: Melhor F1 = 0.9332
Geração 8: Melhor F1 = 0.9332


In [ ]:
%pip install openai

In [ ]:
%pip install openai azure-ai-ml azure-identity pandas scikit-learn matplotlib seaborn

In [ ]:
from openai import AzureOpenAI
import os

# 1. Ajuste o Endpoint para ser APENAS a base (remova o caminho do deployment)
# 2. Use a versão de API estável
client = AzureOpenAI(
    api_key="Sua_Chave_de_API_Aqui",  
    api_version="2024-08-01-preview", 
    azure_endpoint="Sua_URL_Base_do_Azure_Aqui"
)

def gerar_explicacao_medica(diagnostico, probabilidade, metricas):
    # IMPORTANTE: Este nome deve ser exatamente o "Nome da Implantação" 
    # que aparece na lista do seu Azure AI Studio.
    # Pela sua URL anterior, parece ser "gpt-4o"
    NOME_DEPLOYMENT = "gpt-4o" 
    
    # Aplicando Prompt Engineering para contexto médico conforme o requisito [cite: 43]
    prompt = f"""
    Você é um assistente médico especialista em oncologia auxiliando na interpretação de modelos de IA.
    
    DADOS DO MODELO:
    - Predição: {'Maligno (Alto Risco)' if diagnostico == 1 else 'Benigno (Baixo Risco)'}
    - Confiança: {probabilidade:.2%}
    - Parâmetros da Otimização Genética: {metricas}
    
    TAREFAS:
    1. Gere uma explicação em linguagem natural do diagnóstico[cite: 40].
    2. Transforme os dados estatísticos em insights acionáveis para a equipe médica[cite: 41].
    3. Sugira os próximos passos clínicos baseados no contexto de câncer de mama.
    """
    
    try:
        response = client.chat.completions.create(
            model=NOME_DEPLOYMENT, 
            messages=[
                {"role": "system", "content": "Você é um assistente médico altamente qualificado."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7 # Adiciona criatividade controlada para a explicação
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erro na conexão: {str(e)}"

# Execução do Teste
# print("Experimento 1")
# print(gerar_explicacao_medica(1, 0.985, exp1["best_params"]))
# print("Experimento 2")
# print(gerar_explicacao_medica(1, 0.985, exp2["best_params"]))
print("Experimento 3")
print(gerar_explicacao_medica(1, 0.985, exp3["best_params"]))




In [11]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

print("Preparando o pacote de treinamento (Job)...")

# 1. Pegamos os melhores hiperparâmetros do seu Experimento 3 (Robusto)
# Digamos que foram 40, 10 e 0.1 (ajuste se os seus foram ligeiramente diferentes)
best_n = int(exp3["best_params"][0])
best_depth = int(exp3["best_params"][1])
best_split = exp3["best_params"][2]

# 2. Definimos o comando exato que vai rodar no servidor da nuvem
job_command = f"""
python train.py \
--dataset ${{{{inputs.dados_hospital}}}} \
--n_estimators {best_n} \
--max_depth {best_depth} \
--min_samples_split {best_split}
"""

# 3. Empacotamos tudo usando o SDK da Azure
job = command(
    code="./",  # Avisa a Azure que o train.py está nesta mesma pasta
    command=job_command,
    # Usamos um ambiente oficial da Microsoft que já vem com Scikit-Learn e MLflow instalados
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    inputs={
        # Conectamos o dataset que você já tinha subido na primeira célula do projeto!
        "dados_hospital": Input(
            type=AssetTypes.URI_FILE,
            path="azureml:breast_cancer_v2:1"
        )
    },
    experiment_name="Treinamento_Oficial_Producao",
    display_name="Treino_Final_Câncer_Mama"
)

# 4. O Lançamento 🚀
print("Enviando o Job para os servidores da Azure...")
trabalho_enviado = ml_client.jobs.create_or_update(job)

print(f"✅ SUCESSO! O Job foi lançado.")
print(f"Acompanhe o treinamento ao vivo clicando aqui: {trabalho_enviado.studio_url}")

Preparando o pacote de treinamento (Job)...
Enviando o Job para os servidores da Azure...
✅ SUCESSO! O Job foi lançado.
Acompanhe o treinamento ao vivo clicando aqui: https://ml.azure.com/runs/honest_rat_s1yvxt20nw?wsid=/subscriptions/d91397ec-f486-481e-a7ba-4e64f05b7962/resourcegroups/meugrupo/workspaces/tech-challenge-2&tid=c6dd0654-4055-40ff-ae95-b68907d70ff3


Uploading gabrielmd908 (0.04 MBs): 100%|██████████| 36172/36172 [00:00<00:00, 344259.89it/s]


